In [1]:
import os

import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, utils

from tqdm import tqdm
from PIL import Image
from glob import glob
import importlib

import models as dit_models
importlib.reload(dit_models)
DiT = dit_models.DiT


In [2]:
class Config:
    device = "cuda" if torch.cuda.is_available() else "cpu"

    image_size = 64
    channels = 3
    patch_size = 4

    dim = 256
    depth = 6
    heads = 8
    mlp_dim = 512
    k = 256

    timesteps = 500
    beta_start = 1e-4
    beta_end = 2e-2
    betas = torch.linspace(beta_start, beta_end, timesteps, device=device)
    alphas = 1.0 - betas
    alpha_bars = torch.cumprod(alphas, dim=0)

    batch_size = 64
    n_epoch = 300
    lr = 1e-4

    save_dir_weight = "dit_weight"
    save_dir_img = "dit_img"
    data_path = "../ddpm/datas/tinyhero"

    os.makedirs(save_dir_weight, exist_ok=True)
    os.makedirs(save_dir_img, exist_ok=True)

config = Config()
config.device


'cuda'

In [3]:
def q_sample(x, t_index, noise):
    sqrt_ab = torch.sqrt(config.alpha_bars[t_index])[:, None, None, None]
    sqrt_mab = torch.sqrt(1.0 - config.alpha_bars[t_index])[:, None, None, None]
    return sqrt_ab * x + sqrt_mab * noise

def t_to_model_input(t_index):
    return t_index.float()

@torch.no_grad()
def sample(model, imgs=None, n=16):
    model.eval()
    if imgs is None:
        imgs = torch.randn(
            n,
            config.channels,
            config.image_size,
            config.image_size,
            device=config.device,
        )
    else:
        n = len(imgs)

    for t in reversed(range(1, config.timesteps)):
        t_batch = torch.full((n,), t, device=config.device, dtype=torch.long)
        noise_pred = model(imgs, t_to_model_input(t_batch))

        beta_t = config.betas[t]
        alpha_t = config.alphas[t]
        alpha_bar_t = config.alpha_bars[t]
        noise = torch.randn_like(imgs) if t > 1 else torch.zeros_like(imgs)

        imgs = (
            (imgs - ((1 - alpha_t) / torch.sqrt(1 - alpha_bar_t)) * noise_pred) / torch.sqrt(alpha_t)
            + torch.sqrt(beta_t) * noise
        )

    t_batch = torch.zeros((n,), device=config.device, dtype=torch.long)
    noise_pred = model(imgs, t_to_model_input(t_batch))
    alpha_t = config.alphas[0]
    alpha_bar_t = config.alpha_bars[0]
    imgs = (
        (imgs - ((1 - alpha_t) / torch.sqrt(1 - alpha_bar_t)) * noise_pred) / torch.sqrt(alpha_t)
    )

    imgs = (imgs.clamp(-1, 1) + 1) / 2
    return imgs


In [4]:
class HeroDataset(Dataset):
    def __init__(self, root, transform=None):
        self.paths = sorted(glob(os.path.join(root, "*/*.png")))
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        img = Image.open(self.paths[idx]).convert("RGBA")
        bg = Image.new("RGBA", img.size, (255, 255, 255, 255))
        img = Image.alpha_composite(bg, img).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img

transform = transforms.Compose([
    transforms.Resize((config.image_size, config.image_size)),
    transforms.ToTensor(),
    transforms.Normalize([0.5] * 3, [0.5] * 3),
])

dataset = HeroDataset(config.data_path, transform)
loader = DataLoader(
    dataset,
    batch_size=config.batch_size,
    shuffle=True,
    num_workers=4,
    pin_memory=True,
)

len(loader), len(dataset)


(57, 3648)

In [5]:
def train():
    model = DiT(
        img_size=config.image_size,
        dim=config.dim,
        patch_size=config.patch_size,
        depth=config.depth,
        heads=config.heads,
        mlp_dim=config.mlp_dim,
        k=config.k,
        in_channels=config.channels,
    ).to(config.device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=config.lr)
    schedule = torch.optim.lr_scheduler.StepLR(optimizer, step_size=60, gamma=0.5)

    print(f"parameters: {sum(p.numel() for p in model.parameters()):,}")

    for epoch in range(1, config.n_epoch + 1):
        model.train()
        pbar = tqdm(loader, desc=f"Epoch {epoch} / {config.n_epoch}")

        for x in pbar:
            x = x.to(config.device)
            bs = x.size(0)

            t_index = torch.randint(0, config.timesteps, (bs,), device=config.device)
            noise = torch.randn_like(x)
            x_t = q_sample(x, t_index, noise)
            noise_pred = model(x_t, t_to_model_input(t_index))
            loss = F.mse_loss(noise_pred, noise)

            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

            pbar.set_postfix({"loss": f"{loss.item():.6f}"})

        schedule.step()
        imgs = sample(model, n=16)
        utils.save_image(imgs, f"{config.save_dir_img}/dit_epoch_{epoch:03d}.png", nrow=4)

        if epoch % 30 == 0:
            torch.save(model.state_dict(), f"{config.save_dir_weight}/dit_{epoch}epoch.pth")

    torch.save(model.state_dict(), f"{config.save_dir_weight}/dit.pth")
    print("Training Complete")
    return model


In [6]:
model = train()


parameters: 6,664,752


Epoch 300 / 300: 100%|██████████| 57/57 [00:07<00:00,  7.21it/s, loss=0.034486]


Training Complete
